# Native AHR v5: coherent search and bounded tactical evidence

This experiment keeps the native Queen, Worker, Scout and pheromone model from v4. Queen compares one completed common-depth set of candidate investigations. Worker checks explicit capture and group-survival obligations with the real rules; Scout proposes forcing replies and reconsiders overlooked root moves. An incomplete deeper round never replaces completed shallower evidence.

**Inputs:** tagged definitions in `00_shared.ipynb` and frozen v4 notebooks; the fixed first AHR-only initialization; development failure records for regression checks. **Outputs:** exported model/search/loss definitions and `reports/ahr_native_v5/model_search_validation.json`. No training or final evaluation runs here. **Runtime:** normally under two minutes on GPU 1. **Checkpoints:** model training checkpoints belong to notebook 03, under `checkpoints/<run_id>/`. **Resume:** Run All from a fresh `ahr-go` kernel; this notebook has no persistent optimizer state. The downstream trainer imports only cells tagged `ahr-export`.

All learned utilities and restricted-branch search results remain UNKNOWN as full-game claims. A bounded Worker SAT/UNSAT claim names its exact predicate and independent rule audit.

In [ ]:
from pathlib import Path
import os, json, nbformat
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebooks/01_go_rules_and_independent_verifier.ipynb').exists())
_shared = nbformat.read(PROJECT/'experiments/ahr_native_v5/00_shared.ipynb', 4)
for _i, _cell in enumerate(_shared.cells):
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags', []):
        exec(compile(_cell.source, f'v5 shared export {_i}', 'exec'), globals())
_ = v5_load_v4(PROJECT, globals())

## Preserve the native AHR model and teach verified candidate distinctions

The architecture and initialization tensors are unchanged. Goal-conditioned successful and unsuccessful legal continuations add a small ranking loss to the existing tactical loss. Cutoffs provide no negative label. Global policy supervision still requires an explicit policy weight supplied by the trainer.

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
import copy, math, time
import numpy as np
import torch
from torch.nn import functional as F

V5AHR = V4AHR

def v5_model_initialization(project, device='cuda:1'):
    model, provenance = v4_model_initialization(project, device)
    return model, {**provenance, 'kind':'native_ahr_v5', 'native_architecture_changed':False,
        'jepa':False, 'external_engine_policy_labels':False,
        'search_teacher':'coherent bounded AHR continuations only'}

def v5_tactical_support(example):
    success = example.get('tactical_success_actions')
    if success is None:
        success = [i for i,p in enumerate(example['tactical_policy_target']) if p>0]
    failure = list(example.get('tactical_failure_actions', []))
    action = example.get('tactical_action')
    result = example.get('tactical_result_target')
    if action is not None and result in (0, 0., False): failure.append(int(action))
    if action is not None and result in (1, 1., True): success = list(success)+[int(action)]
    success, failure = sorted(set(success)), sorted(set(failure))
    if set(success)&set(failure): raise ValueError('A verified continuation cannot be both success and failure')
    return success, failure

def v5_learning_loss(model, examples):
    loss, metrics = v4_learning_loss(model, examples)
    tactical = [e for e in examples if 'tactical_goal' in e]
    terms=[]
    if tactical:
        states=[GoState.from_dict(e['state']) for e in tactical]
        output=model.tactical_forward(states,[e['tactical_goal'] for e in tactical],[e['tactical_targets'] for e in tactical])
        for i,e in enumerate(tactical):
            success,failure=v5_tactical_support(e)
            legal=set(legal_actions(states[i]))
            if any(a not in legal for a in success+failure):raise ValueError('Illegal tactical candidate supervision')
            if success and failure:
                # Energies are comparable within one explicit goal and target mask only.
                good=output['policy_logits'][i,success].mean()
                bad=output['policy_logits'][i,failure].mean()
                terms.append(F.softplus(1.+bad-good))
    margin=torch.stack(terms).mean() if terms else loss*0
    loss=loss+.1*model.config.tactical_weight*margin
    metrics.update(loss=float(loss.detach()),tactical_margin=float(margin.detach()),tactical_margin_examples=len(terms))
    if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite native AHR v5 loss')
    return loss,metrics

def v5_predict_tactical(model, examples, batch_size=64):
    if not isinstance(model,V4AHR) or not examples:raise ValueError('Model and nonempty verified tasks required')
    previous=model.training;model.eval();rows=[]
    try:
        for start in range(0,len(examples),batch_size):
            batch=examples[start:start+batch_size];states=[GoState.from_dict(e['state']) for e in batch]
            with torch.inference_mode():
                logits=model.tactical_forward(states,[e['tactical_goal'] for e in batch],[e['tactical_targets'] for e in batch])['policy_logits']
                probs=logits.softmax(-1).detach().cpu().numpy()
            for e,p in zip(batch,probs):
                success,_=v5_tactical_support(e);chosen=int(np.argmax(p))
                rows.append({'goal':e['tactical_goal'],'family_id':e.get('family_id'),'action':chosen,
                    'correct':chosen in success,'success_probability_mass':float(sum(p[a] for a in success))})
    finally:model.train(previous)
    by_goal={goal:{'count':sum(r['goal']==goal for r in rows),
        'top1_success':float(np.mean([r['correct'] for r in rows if r['goal']==goal])),
        'probability_mass_success':float(np.mean([r['success_probability_mass'] for r in rows if r['goal']==goal]))}
        for goal in sorted({r['goal'] for r in rows})}
    return {'count':len(rows),'top1_success':float(np.mean([r['correct'] for r in rows])),
        'probability_mass_success':float(np.mean([r['success_probability_mass'] for r in rows])),
        'by_goal':by_goal,'rows':rows,'scope':'Heldout exact bounded goal-conditioned tasks; not full-game strength'}


## Explicit bounded Worker obligations

Capture is checked against the actual root successor. For survival over the root move and one opponent reply, Worker enumerates every legal capture point of every resulting group in atari. Any immediate capture must fill such a sole liberty, so excluding all those counterexamples exhausts this exact one-reply predicate. A checked counterexample means UNSAT for that survival predicate, not a proved full-game loss. Every emitted witness is checked by the independent transition verifier.

The trained tactical head now connects to the move search: Scout queries each urgent goal and target mask, proposes actions from its logits, and adds a small prior. Every admitted proposal receives the same completed search depth. A learned probability never changes the independently verified Worker certificate.

The predeclared conservative penalty is `-0.60 × tanh(max(0, proved next-reply loss − proved same-move captures) / 3)`. One uncompensated lost stone costs about 0.193 utility units. The penalty is bounded by 0.60 and all legal moves remain available: learned continuation utility can outweigh it. Only checked captures compensate a material exchange. This is a development heuristic, not a theorem that every threatened group should be saved.

In [ ]:
def v5_worker_obligations(state, child, action):
    if child.move_number!=state.move_number+1 or child.to_play!=-state.to_play:
        raise ValueError('Worker requires one exact root successor')
    if not verify_go_emission(state,action,child):raise RuntimeError('Independent Worker root emission audit rejected successor')
    color=state.to_play
    original={p for p,c in enumerate(state.board) if c==color}
    captures=sum(c==-color for c in state.board)-sum(c==-color for c in child.board)
    own_after={p for p,c in enumerate(child.board) if c==color}
    relevant=set()
    for stones,libs in groups_and_liberties(child.board,child.size):
        if child.board[next(iter(stones))]==color and len(libs)==1:relevant.update(libs)
    counterexamples=[];largest_loss=len(original-own_after)
    if not child.terminal:
        legal=set(legal_actions(child))
        for action in sorted(relevant&legal):
            reply=play(child,action)
            if not verify_go_emission(child,action,reply):raise RuntimeError('Independent Worker witness audit rejected reply')
            lost=sorted(p for p in original if reply.board[p]!=color)
            all_lost=sorted(p for p in own_after if reply.board[p]!=color)
            largest_loss=max(largest_loss,len(all_lost))
            if lost:counterexamples.append({'opponent_action':action,'lost_original_stones':lost,'lost_after_move_stones':all_lost})
    survives=not (original-own_after) and not counterexamples
    return {'capture':{'status':'SAT' if captures>0 else 'UNSAT','captured_stones':int(captures),
                'predicate':'The selected root move captures at least one opponent stone'},
        'survival':{'status':'SAT' if survives else 'UNSAT','target_stones':sorted(original),
            'predicate':'Every original own stone survives this move and every legal single opponent reply',
            'relevant_capture_points_exhausted':True,'capture_points_checked':sorted(relevant),
            'counterexamples':counterexamples},
        'max_immediate_loss':int(largest_loss),'scope':'Exactly two plies; no full-game life or loss claim'}

def v5_scout_task_proposals(model,state,maximum_tasks=2,topk=2):
    if model.training or not isinstance(model,V4AHR):raise ValueError('Scout needs an evaluation-mode native AHR model')
    tasks=v4_tactical_tasks(state,maximum=maximum_tasks)
    if not tasks:return {'proposals':[],'tasks':[],'probabilities':[0.]*82,'scope':'Learned task priors only; no bounded certificate'}
    with torch.inference_mode():
        output=model.tactical_forward([state]*len(tasks),[t[1] for t in tasks],[t[2] for t in tasks])
        probabilities=output['policy_logits'].softmax(-1).detach().cpu().numpy()
    if not np.isfinite(probabilities).all():raise FloatingPointError('Nonfinite goal-conditioned Scout prior')
    legal=list(legal_actions(state));proposals=[];records=[]
    for task,prior in zip(tasks,probabilities):
        selected=sorted(legal,key=lambda a:(-prior[a],a))[:topk]
        proposals.extend(selected)
        records.append({'goal':task[1],'target_stones':task[2],'proposed_actions':selected,
            'selected_probabilities':[float(prior[a]) for a in selected],
            'status':'UNKNOWN','interpretation':'Goal-conditioned learned ordering, not a verified successful action'})
    return {'proposals':list(dict.fromkeys(proposals)),'tasks':records,'probabilities':probabilities.max(0).tolist(),
        'scope':'Learned task priors only; every admitted proposal requires the same-depth rule search'}

@dataclass(frozen=True)
class V5SearchConfig:
    time_budget_s:float=1.
    queen_cycles:int=2
    max_depth:int=4
    max_nodes:int=768
    branch_width:int=8
    tactical_width:int=6
    quiescence_extensions:int=0
    policy_temperature:float=.15
    score_weight:float=.25
    ownership_weight:float=.10
    static_weight:float=.40
    prior_weight:float=.015
    tactical_prior_weight:float=.025
    reserve_fraction:float=.20
    baseline_replies:int=6
    reconsideration_width:int=4
    immediate_loss_weight:float=.60
    goal_tasks:int=2
    goal_proposals_per_task:int=2
    def __post_init__(self):
        # Reuse audited scalar/integer bounds and reject unsupported work limits.
        V4SearchConfig(**{k:v for k,v in asdict(self).items() if k in V4SearchConfig.__dataclass_fields__})
        if type(self.baseline_replies)is not int or not 1<=self.baseline_replies<=82:raise ValueError('Invalid baseline reply width')
        if type(self.reconsideration_width)is not int or not 1<=self.reconsideration_width<=82:raise ValueError('Invalid reconsideration width')
        if type(self.goal_tasks)is not int or not 0<=self.goal_tasks<=4:raise ValueError('Invalid bounded task count')
        if type(self.goal_proposals_per_task)is not int or not 1<=self.goal_proposals_per_task<=8:raise ValueError('Invalid Scout proposal count')
        if not math.isfinite(self.immediate_loss_weight) or self.immediate_loss_weight<0:raise ValueError('Invalid immediate loss weight')

class V5SearchCutoff(Exception):pass

def v5_choose_move(model, state, search_config=None):
    config=V5SearchConfig(**search_config) if isinstance(search_config,dict) else search_config or V5SearchConfig()
    if not isinstance(config,V5SearchConfig) or not isinstance(model,V4AHR) or not isinstance(state,GoState) or state.size!=9 or model.training:
        raise ValueError('Evaluation-mode native AHR and exact 9x9 GoState required')
    start=time.monotonic();deadline=start+config.time_budget_s*(1-config.reserve_fraction)
    device=next(model.parameters()).device;cache={};group_cache={};feature_cache={}
    stats={'nodes':0,'model_calls':0,'model_positions':0,'queen_iterations':0,'worker_transitions':0,
        'scout_candidates':0,'alpha_beta_prunes':0,'tactical_model_calls':0,'selective_extensions':0,'goal_scout_candidates':0}
    def check():
        if time.monotonic()>=deadline:raise V5SearchCutoff('time_budget')
        if stats['nodes']>=config.max_nodes:raise V5SearchCutoff('node_budget')
    def groups(position):
        key=v4_position_key(position)
        if key not in group_cache:group_cache[key]=groups_and_liberties(position.board,position.size)
        return group_cache[key]
    def child(position,action):
        check();stats['worker_transitions']+=1;stats['nodes']+=1;return play(position,action)
    def feats(position,action,result):
        key=(v4_position_key(position),action)
        if key not in feature_cache:feature_cache[key]=v4_move_features(position,action,result,groups(position))
        return feature_cache[key]
    def evaluate(positions):
        pending=[];seen=set()
        for position in positions:
            key=v4_position_key(position)
            if key not in cache and key not in seen and not position.terminal:pending.append((key,position));seen.add(key)
        if not pending:return
        check()
        with torch.inference_mode():out=model(encode_go_batch([p for _,p in pending],device),queen_cycles=config.queen_cycles)
        values=out['value'].detach().cpu().tolist();scores=out['score'].detach().cpu().tolist()
        owners=out['ownership'].detach().mean(-1).cpu().tolist();logits=out['policy_logits'].detach().cpu().numpy()
        if not all(math.isfinite(x) for x in values+scores+owners) or np.isnan(logits).any():raise FloatingPointError('Nonfinite AHR inference')
        for i,(key,position) in enumerate(pending):
            prior=np.exp(logits[i]-np.max(logits[i]));prior/=prior.sum()
            utility=((1-config.score_weight-config.ownership_weight-config.static_weight)*values[i]+
                config.score_weight*math.tanh(scores[i]/20)+config.ownership_weight*owners[i]+
                config.static_weight*math.tanh(v4_static_score(position)/20))
            cache[key]={'utility':float(utility),'value':float(values[i]),'prior':prior}
        stats['model_calls']+=1;stats['model_positions']+=len(pending)
    def utility(position):
        if position.terminal:
            relative=position.to_play*area_score(position)
            return float((relative>0)-(relative<0))
        evaluate([position]);return cache[v4_position_key(position)]['utility']
    def children(position,width):
        check();evaluate([position]);prior=cache[v4_position_key(position)]['prior'];legal=list(legal_actions(position))
        forcing=set()
        # No final [:width] truncation: all sole-liberty saves/captures and two-liberty continuations are retained.
        for stones,libs in groups(position):
            if len(libs)<=2:forcing.update(libs)
        forcing.intersection_update(legal)
        ranked=sorted(legal,key=lambda a:(a not in forcing,-prior[a],a))
        chosen=list(sorted(forcing))+[a for a in ranked if a not in forcing][:width]
        result=[]
        for action in chosen:
            successor=child(position,action);result.append((action,successor,feats(position,action,successor)))
        result.sort(key=lambda x:(-x[2]['priority'],-prior[x[0]],x[0]))
        stats['scout_candidates']+=len(result);return result
    def visit(position,depth,alpha=-float('inf'),beta=float('inf')):
        check()
        if position.terminal:return utility(position)
        if depth<=0:return utility(position)
        replies=children(position,config.tactical_width)
        evaluate([p for _,p,_ in replies]);best=-float('inf')
        for _,successor,_ in replies:
            value=-visit(successor,depth-1,-beta,-alpha);best=max(best,value);alpha=max(alpha,value)
            if alpha>=beta:stats['alpha_beta_prunes']+=1;break
        return best
    base={'strategic_status':'UNKNOWN','verified_legal':True,'jepa':False,'status':'UNKNOWN',
        'utility_scope':'Learned estimates and restricted AHR continuations; not a full-game proof',
        'immediate_loss_weight':config.immediate_loss_weight,'loss_calibration':'Bounded -.60*tanh(max(0,proved immediate loss - proved same-move captures)/3) at default weight; all legal sacrifices remain eligible',
        'policy_target_kind':'coherent_completed_ahr_evidence','root_rounds_completed':0,'coherent_root_rounds':True}
    if state.terminal:
        return {**base,'status':'TERMINAL','action':None,'policy':[0.]*82,'policy_target':[0.]*82,
            'root_value':utility(state),'root_neural_value':None,'completed_depth':0,'comparison_depth':0,
            'policy_weight':0.,'policy_label_eligible':False,'comparison_complete':False,'elapsed_s':time.monotonic()-start,'deadline_missed':False,**stats}
    legal=list(legal_actions(state));prior=np.zeros(82);prior[legal]=1/len(legal)
    chosen=state.pass_action;neural_value=None;reason='zero_budget';root_children={};root_features={};workers={}
    shallow={};committed={};depth=0;rounds=0;reconsidered=[];unfinished_round=False
    scout_tasks=[];goal_proposals=[];goal_bonus=np.zeros(82)
    if state.consecutive_passes==1 and config.time_budget_s>0:
        terminal=play(state,state.pass_action)
        if not verify_go_emission(state,state.pass_action,terminal) or area_score(terminal)!=arena_reference_score(terminal):raise RuntimeError('Independent terminal-pass audit disagrees')
        if state.to_play*area_score(terminal)>0:
            p=[0.]*82;p[81]=1.;elapsed=time.monotonic()-start
            return {**base,'status':'VERIFIED_TERMINAL_WIN','action':81,'policy':p,'policy_target':p,'policy_weight':1.,'policy_label_eligible':True,'comparison_complete':True,
                'root_value':1.,'root_neural_value':None,'completed_depth':1,'comparison_depth':1,'root_depths':{'81':1},
                'root_rounds_completed':1,'reason':'verified_terminal_win','elapsed_s':elapsed,'deadline_missed':elapsed>config.time_budget_s,**stats}
    if config.time_budget_s>0:
        try:
            evaluate([state]);root=cache[v4_position_key(state)];prior=root['prior'];neural_value=root['value']
            chosen=max(legal,key=lambda a:(prior[a],-a));reason='native_prior'
            # Complete the cheap root Scout pass before comparing any root choices.
            for action in legal:
                successor=child(state,action);root_children[action]=successor
                root_features[action]=feats(state,action,successor)
                check();workers[action]=v5_worker_obligations(state,successor,action)
            evaluate(list(root_children.values()))
            def bonus(action):
                feature=root_features[action];risk=workers[action]['max_immediate_loss']
                compensated=workers[action]['capture']['captured_stones']
                unforced_loss=max(0,risk-compensated)
                return .16*math.tanh(feature['priority']/4)-config.immediate_loss_weight*math.tanh(unforced_loss/3)+config.tactical_prior_weight*goal_bonus[action]
            for action,successor in root_children.items():shallow[action]=-utility(successor)+bonus(action)
            committed=shallow.copy();depth=1;rounds=1;chosen=max(committed,key=lambda a:(committed[a],prior[a],-a));reason='completed_root_scout'
            if config.goal_tasks and config.tactical_prior_weight:
                check();scout=v5_scout_task_proposals(model,state,config.goal_tasks,config.goal_proposals_per_task)
                scout_tasks=scout['tasks'];goal_proposals=scout['proposals'];goal_bonus=np.asarray(scout['probabilities'])
                if scout_tasks:stats['tactical_model_calls']+=1
                stats['goal_scout_candidates']=len(goal_proposals)
                # Only the completed shallow estimates are recomputed; Worker truth is unchanged by a learned prior.
                shallow={a:-utility(root_children[a])+bonus(a) for a in root_children}
                committed=shallow.copy();chosen=max(committed,key=lambda a:(committed[a],prior[a],-a))
            proposal=sorted(legal,key=lambda a:(-shallow[a],-prior[a],a))
            urgent=sorted([a for a in legal if root_features[a]['captures'] or root_features[a]['saved_stones']],
                key=lambda a:(-root_features[a]['priority'],-shallow[a],a))
            pool=list(dict.fromkeys(urgent+goal_proposals+proposal[:config.branch_width]))
            # Every candidate admitted to the comparison has the same complete two-ply search.
            staged={};reply_map={}
            for action in pool:
                successor=root_children[action]
                reply_map[action]=[] if successor.terminal else children(successor,config.baseline_replies)
            evaluate([p for replies in reply_map.values() for _,p,_ in replies])
            for action,replies in reply_map.items():
                staged[action]=((-utility(root_children[action])) if not replies else min(utility(p) for _,p,_ in replies))+bonus(action)
            committed=staged;depth=2;rounds+=1;chosen=max(committed,key=lambda a:(committed[a],prior[a],-a));reason='completed_common_depth_round'
            # Scout revisits low-prior and initially overlooked choices at the same committed depth.
            for action in [a for a in proposal if a not in committed][:config.reconsideration_width]:
                check();replies=children(root_children[action],config.baseline_replies)
                evaluate([p for _,p,_ in replies]);value=min(utility(p) for _,p,_ in replies) if replies else -utility(root_children[action])
                committed[action]=value+bonus(action);reconsidered.append(action)
                chosen=max(committed,key=lambda a:(committed[a],prior[a],-a))
            # Deeper rounds are transactional across the declared pool. Partial work is only a diagnostic.
            for next_depth in range(3,config.max_depth+1):
                check();stats['queen_iterations']+=1
                frontier=sorted(committed,key=lambda a:(-committed[a],-prior[a],a))[:config.branch_width]
                staged={};unfinished_round=True
                for action in frontier:
                    staged[action]=-visit(root_children[action],next_depth-1)+bonus(action)
                committed=staged;depth=next_depth;rounds+=1;unfinished_round=False
                chosen=max(committed,key=lambda a:(committed[a],prior[a],-a))
                reason='completed_common_depth_round'
        except V5SearchCutoff as cutoff:reason=str(cutoff)
    policy=np.zeros(82,dtype=float)
    if committed:
        moves=sorted(committed);values=np.asarray([committed[a] for a in moves])
        logits=(values-values.max())/config.policy_temperature+config.prior_weight*np.log(np.maximum(prior[moves],1e-8))
        weights=np.exp(logits-logits.max());weights/=weights.sum();policy[moves]=weights
        root_value=float(committed[chosen]);policy_weight=1. if depth>=2 else 0.
    else:
        policy[legal]=prior[legal];policy/=policy.sum();root_value=neural_value;policy_weight=0.
    successor=play(state,chosen)
    if not verify_go_emission(state,chosen,successor):raise RuntimeError('Independent final emission audit rejected move')
    elapsed=time.monotonic()-start
    return {**base,'action':int(chosen),'policy':policy.tolist(),'policy_target':policy.tolist(),'policy_weight':policy_weight,
        'policy_label_eligible':bool(policy_weight),'comparison_complete':bool(depth>=2),'root_value':root_value,'root_neural_value':neural_value,'completed_depth':depth,'comparison_depth':depth,
        'root_depths':{str(a):depth for a in committed},'root_utilities':{str(a):float(v) for a,v in committed.items()},'root_rounds_completed':rounds,'completed_root_checks':len(committed) if depth>=2 else 0,
        'root_candidates':sorted(committed),'root_tactical_candidates':[a for a,f in root_features.items() if f['captures'] or f['saved_stones']],
        'root_worker_receipts':{str(a):workers[a] for a in committed},'worker_tasks':len(workers),
        'goal_scout_proposals':goal_proposals,'goal_scout_tasks':scout_tasks,'goal_scout_scope':'Learned priors, never bounded certificates',
        'unreviewed_root_actions':[a for a in legal if a not in committed],
        'unreviewed_status':'UNKNOWN','reconsidered_root_actions':reconsidered,'unfinished_deeper_round_discarded':unfinished_round,
        'comparison_scope':'Only the last completed common-depth root candidate pool; root admissions receive the same depth',
        'fallback_pass':not bool(committed) and chosen==81,'reason':reason,'elapsed_s':elapsed,'deadline_missed':elapsed>config.time_budget_s,**stats}


## Regression checks and visible receipts

These checks establish native tensor identity, separate tactical gradients, coherent common-depth comparisons, exact capture/survival certificates, cutoff UNKNOWN behavior, independent legality and warmed one-second timing. They replay a previously lost group from the frozen development match. Passing does not establish GNU Go win strength.

In [ ]:
def v5_validate_model_search(project, device='cuda:1'):
    v4_seed(2026100205)
    model,provenance=v5_model_initialization(project,device)
    original,old_provenance=v4_model_initialization(project,device)
    assert nested_equal(model.state_dict(),original.state_dict())
    states=[initial_state(9,7.5)]
    board=[0]*81;board[40]=-1
    for p in (31,39,49):board[p]=1
    capture=GoState(9,tuple(board),to_play=1)
    board=[0]*81;board[40]=1
    for p in (31,39,49):board[p]=-1
    save=GoState(9,tuple(board),to_play=1)
    states.extend([capture,save])
    # Load development actions only; final/lockbox positions remain untouched.
    failure_path=Path(project)/'reports/ahr_native_v4/development/ahr-native-v4-dev-g0256-results.json'
    if failure_path.exists():
        failure=json.loads(failure_path.read_text());game=failure['results'][0]
        replay=initial_state(9,7.5)
        for action in game['actions'][:18]:replay=play(replay,action)
        states.append(replay)
    with torch.inference_mode():model(encode_go_batch(states,device))
    # The first larger forward warms shape-dependent kernels without entering the timed API.
    with torch.inference_mode():model(encode_go_batch([states[0]]*96,device))
    decisions=[]
    for name,position in zip(['opening','one_ply_capture','two_ply_survival','frozen_development_failure'],states):
        receipt=v5_choose_move(model,position,V5SearchConfig())
        assert receipt['action'] in legal_actions(position)
        assert verify_go_emission(position,receipt['action'],play(position,receipt['action']))
        assert abs(sum(receipt['policy'])-1)<1e-8
        assert all(p==0 for a,p in enumerate(receipt['policy']) if a not in legal_actions(position))
        assert set(receipt.get('root_depths',{}).values()) in (set(),{receipt['completed_depth']})
        assert receipt['coherent_root_rounds'] and receipt['strategic_status']=='UNKNOWN'
        assert not receipt['deadline_missed'],(name,receipt['elapsed_s'])
        if name=='one_ply_capture':assert receipt['action']==41,receipt['action']
        if name=='two_ply_survival':assert receipt['action']==41,receipt['action']
        decisions.append({'case':name,**{k:receipt[k] for k in ('action','completed_depth','comparison_depth','root_rounds_completed','nodes','model_positions','elapsed_s','deadline_missed')},
            'reconsidered':receipt['reconsidered_root_actions'],'unfinished_round_discarded':receipt['unfinished_deeper_round_discarded']})
    cap=v5_worker_obligations(capture,play(capture,41),41);assert cap['capture']['status']=='SAT'
    survives=v5_worker_obligations(save,play(save,41),41);assert survives['survival']['status']=='SAT'
    loses=v5_worker_obligations(save,play(save,0),0);assert loses['survival']['status']=='UNSAT'
    assert loses['survival']['counterexamples'][0]['opponent_action']==41
    rejected=False
    try:v5_worker_obligations(capture,play(capture,41),0)
    except RuntimeError:rejected=True
    assert rejected,'A malformed root successor must never receive a Worker certificate'
    zero=v5_choose_move(model,states[0],V5SearchConfig(time_budget_s=0))
    assert zero['status']=='UNKNOWN' and zero['policy_weight']==0 and zero['action'] in legal_actions(states[0])
    low=v5_choose_move(model,states[0],V5SearchConfig(max_nodes=1))
    assert low['status']=='UNKNOWN' and low['policy_weight']==0
    terminal=v5_choose_move(model,play(states[0],81),V5SearchConfig())
    assert terminal['status']=='VERIFIED_TERMINAL_WIN' and terminal['action']==81
    # Controlled head-output injection tests the bridge, not tactical learning quality.
    before=v5_worker_obligations(capture,play(capture,41),41)
    saved_forward=model.tactical_forward
    def injected_forward(states,goals,targets,preferred):
        legal=encode_go_batch(states,device)['legal_mask']
        logits=torch.full(legal.shape,-20.,device=device).masked_fill(~legal,-torch.inf)
        logits[:,preferred]=20.
        return {'policy_logits':logits}
    try:
        model.tactical_forward=lambda states,goals,targets:injected_forward(states,goals,targets,0)
        first=v5_scout_task_proposals(model,capture,topk=1)
        model.tactical_forward=lambda states,goals,targets:injected_forward(states,goals,targets,41)
        second=v5_scout_task_proposals(model,capture,topk=1)
    finally:model.tactical_forward=saved_forward
    assert first['proposals']!=second['proposals'] and first['proposals']==[0] and second['proposals']==[41]
    assert nested_equal(before,v5_worker_obligations(capture,play(capture,41),41))
    row={'state':capture.to_dict(),'policy_target':[0.]*82,'policy_weight':0.,'tactical_goal':'capture_one_ply',
        'tactical_targets':[40],'tactical_policy_target':[float(a==41) for a in range(82)],
        'tactical_success_actions':[41],'tactical_failure_actions':[0]}
    model.train();loss,metrics=v5_learning_loss(model,[row]);loss.backward()
    assert metrics['tactical_margin_examples']==1 and torch.isfinite(loss)
    assert model.goal_embedding.weight.grad is not None and model.goal_embedding.weight.grad.abs().sum()>0
    for name,param in model.base.named_parameters():
        if any(k in name for k in ('queen.energy','worker.energy','scout.energy','value_head','score_head','ownership_head')):
            assert param.grad is None or bool((param.grad==0).all()),name
    model.eval();model.zero_grad(set_to_none=True)
    tactical=v5_predict_tactical(model,[row]);assert tactical['count']==1
    return {'status':'PASS','device':device,'initialization':provenance,'native_tensors_equal':True,
        'jepa':False,'decisions':decisions,'coherent_depth':'PASS','exact_worker_capture_survival':'PASS','malformed_worker_root_rejected':'PASS',
        'zero_and_node_cutoff_unknown':'PASS','separate_tactical_margin_gradients':'PASS','terminal_pass':'PASS',
        'goal_conditioned_scout_bridge':'PASS','scout_prior_never_changes_worker_truth':'PASS','loss_metrics':metrics,'untrained_task_prediction':{k:v for k,v in tactical.items() if k!='rows'},
        'sources':v5_sources(project,require_all=False),'strength_claim':'NOT_ESTABLISHED'}


In [ ]:
validation=v5_validate_model_search(PROJECT,os.environ.get('AHRV5_MODEL_VALIDATION_DEVICE','cuda:1'))
_report=PROJECT/'reports/ahr_native_v5/model_search_validation.json'
v4_atomic_json(_report,validation)
from IPython.display import display, HTML
from html import escape
print('Model/search validation:',validation['status'], '| Device:',validation['device'], '| Playing strength:',validation['strength_claim'])
columns=['case','action','comparison_depth','root_rounds_completed','elapsed_s','deadline_missed']
rows=''.join('<tr>'+''.join('<td>'+escape(f"{r[k]:.3f}" if k=='elapsed_s' else str(r[k]))+'</td>' for k in columns)+'</tr>' for r in validation['decisions'])
display(HTML('<table><thead><tr>'+''.join('<th>'+escape(k.replace('_',' '))+'</th>' for k in columns)+'</tr></thead><tbody>'+rows+'</tbody></table>'))
print('Native tensor identity, coherent comparison depths, bounded Worker evidence and independent legality: PASS')

## Frozen failure counterfactual

Replay the first lost-group position from the old development match. Compare the recorded move with fresh v5 decisions at one, three and ten seconds. Filter the independently checked exhaustive capture witnesses to the **first lost group**, containing stone 59. A separate group at stone 69 is also endangered: the stronger predicate that every original own stone survives remains UNSAT. This is one regression position; it is not an estimate of overall game strength.

In [ ]:
_failure_data=json.loads((PROJECT/'reports/ahr_native_v4/development/ahr-native-v4-dev-g0256-results.json').read_text())
_failure=_failure_data['results'][0]
_target_stones=set(_failure_data['failure_analysis']['games'][0]['first_lost_group']['groups'][0]['stones'])
_position=initial_state(9,7.5)
for _action in _failure['actions'][:18]:_position=play(_position,_action)
_recorded=_failure['actions'][18]
_recorded_worker=v5_worker_obligations(_position,play(_position,_recorded),_recorded)
_model,_provenance=v5_model_initialization(PROJECT,os.environ.get('AHRV5_MODEL_VALIDATION_DEVICE','cuda:1'))
with torch.inference_mode():_warmup=_model(encode_go_batch([_position]*96,next(_model.parameters()).device))
_comparisons=[]
for _budget in (1.,3.,10.):
    _decision=v5_choose_move(_model,_position,V5SearchConfig(time_budget_s=_budget,max_nodes=4096,branch_width=10))
    _worker=v5_worker_obligations(_position,play(_position,_decision['action']),_decision['action'])
    _comparisons.append({'budget s':_budget,'action':_decision['action'],'common depth':_decision['completed_depth'],
        'elapsed s':round(_decision['elapsed_s'],3),'deadline missed':_decision['deadline_missed'],
        'first lost group survival':'SAT' if all(play(_position,_decision['action']).board[p]==_position.to_play for p in _target_stones) and not any(_target_stones.intersection(w['lost_original_stones']) for w in _worker['survival']['counterexamples']) else 'UNSAT',
        'all original groups survive':_worker['survival']['status']})
_counterfactual={'status':'PASS' if all(r['first lost group survival']=='SAT' and not r['deadline missed'] for r in _comparisons) else 'OBSERVED_FAILURE',
    'recorded_action':_recorded,'target_stones':sorted(_target_stones),'recorded_survival':_recorded_worker['survival'],'comparisons':_comparisons,
    'sources':v5_sources(PROJECT,require_all=False),
    'scope':'One frozen development capture failure; bounded survival only, no full-game superiority claim'}
v5_atomic_json(PROJECT/'reports/ahr_native_v5/search_counterfactual.json',_counterfactual)
print('Recorded action:',_recorded,'| Original-stone survival:',_recorded_worker['survival']['status'])
print('V5 counterfactual:',_counterfactual['status'])
_headers=list(_comparisons[0])
_table=''.join('<tr>'+''.join('<td>'+escape(str(r[k]))+'</td>' for k in _headers)+'</tr>' for r in _comparisons)
display(HTML('<table><thead><tr>'+''.join('<th>'+escape(k)+'</th>' for k in _headers)+'</tr></thead><tbody>'+_table+'</tbody></table>'))

## Fixed conservative-loss calibration check

Use the same 16 unique-urgency development tasks selected before this calibration: eight captures and eight sole threatened own groups. The previously failed family remains in the set. This tests the fixed 0.60 loss penalty and goal-conditioned proposal bridge together; it does not estimate a full-game win rate. The raw baseline and the original 15/16 outcome remain saved separately. No task or threshold is removed in response to this check.

In [ ]:
_tasks_path=PROJECT/'reports/ahr_native_v5/teacher_baseline_tasks.json'
if _tasks_path.exists():
    _ = v5_import_exports(PROJECT/V5_ROOT/'02_verified_curriculum.ipynb',globals())
    _tasks=json.loads(_tasks_path.read_text())['unique_urgent_tasks']
    _calibration_rows=[]
    for _row in _tasks:
        _state=from_dict(_row['state']);_case={'state':_row['state'],'goal':_row['tactical_goal'],'targets':_row['tactical_targets']}
        _record={'family_id':_row['family_id'],'goal':_row['tactical_goal']}
        for _name,_cfg in [('fast',V5SearchConfig()),('offline',V5SearchConfig(time_budget_s=3.,max_depth=4,max_nodes=2048,branch_width=12))]:
            _choice=v5_choose_move(_model,_state,_cfg);_proof=v5_bounded_action(_case,_choice['action'])
            _record[_name]={'action':_choice['action'],'bounded_status':_proof['status'],'success':_proof['status']=='SAT',
                'comparison_complete':_choice['comparison_complete'],'elapsed_s':_choice['elapsed_s'],
                'deadline_missed':_choice['deadline_missed'],'comparison_depth':_choice['comparison_depth']}
        _calibration_rows.append(_record)
    _fast=sum(r['fast']['success'] for r in _calibration_rows);_offline=sum(r['offline']['success'] for r in _calibration_rows)
    _coherent=all(r['offline']['comparison_complete'] and not r['offline']['deadline_missed'] for r in _calibration_rows)
    _calibration={'status':'PASS' if _coherent and _offline>=_fast and _offline/len(_tasks)>=.95 and (_offline>_fast or _fast/len(_tasks)>=.95) else 'FAIL',
        'count':len(_tasks),'fast_success':_fast,'offline_success':_offline,'coherent_offline':_coherent,
        'fixed_search_config':asdict(V5SearchConfig()),'offline_search_config':asdict(V5SearchConfig(time_budget_s=3.,max_depth=4,max_nodes=2048,branch_width=12)),
        'case_identity_sha256':v5_digest([r['family_id'] for r in _tasks]),'rows':_calibration_rows,'sources':v5_sources(PROJECT,require_all=False),
        'selection':'Same previously saved eight capture and eight sole-own-atari families; no failed task removed',
        'scope':'Fixed development regression set; no full-game strength claim'}
    v5_atomic_json(PROJECT/'reports/ahr_native_v5/search_loss_calibration.json',_calibration)
    print('Fixed 16-task calibration:',_calibration['status'],'| Fast:',str(_fast)+'/16','| Offline:',str(_offline)+'/16')
    _summary=[{'mode':'fast 1 second','verified successes':_fast,'tasks':len(_tasks)},
              {'mode':'offline 3 seconds','verified successes':_offline,'tasks':len(_tasks)}]
    display(HTML('<table><tr><th>Mode</th><th>Verified successes</th><th>Tasks</th></tr>'+''.join('<tr><td>'+r['mode']+'</td><td>'+str(r['verified successes'])+'</td><td>'+str(r['tasks'])+'</td></tr>' for r in _summary)+'</table>'))
else:
    print('Fixed calibration tasks are not yet available; notebook 04 performs the required actual teacher gate.')

The visible validation checks implementation properties. Notebook 04 evaluates unseen tactical families and the fast/offline teacher comparison before notebook 05 can launch the authorized long experiment. Poor pilot metrics remain visible; they are never changed into successful strength claims.